In [ ]:
import sys,subprocess,importlib.util
for p in ["mne","lime","xgboost","catboost"]:
    if importlib.util.find_spec(p) is None: subprocess.check_call([sys.executable,"-m","pip","install","-q",p])
import os,glob,zipfile,time,numpy as np,pandas as pd,matplotlib.pyplot as plt,mne
from sklearn.model_selection import train_test_split,RandomizedSearchCV,StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
from scipy.stats import loguniform
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lime.lime_tabular import LimeTabularExplainer
np.random.seed(42)


In [ ]:
def extract_zip_archives(root="/content"):
    out=os.path.join(root,"sleep_edf_extracted"); os.makedirs(out,exist_ok=True)
    for z in glob.glob(os.path.join(root,"**","*.zip"),recursive=True):
        if zipfile.is_zipfile(z):
            d=os.path.join(out,os.path.splitext(os.path.basename(z))[0]); os.makedirs(d,exist_ok=True)
            with zipfile.ZipFile(z) as f: f.extractall(d)
def find_pair(root="/content"):
    extract_zip_archives(root)
    fs=[p for p in glob.glob(os.path.join(root,"**","*"),recursive=True) if os.path.isfile(p) and p.lower().endswith(".edf")]
    ps=sorted([p for p in fs if "psg" in os.path.basename(p).lower()]); hs=sorted([p for p in fs if "hypnogram" in os.path.basename(p).lower()])
    for p in ps:
        pref=os.path.basename(p)[:6].lower(); m=[h for h in hs if os.path.basename(h).lower().startswith(pref)]
        if m: return p,m[0]
    raise FileNotFoundError("Upload matching PSG and Hypnogram EDF files to /content")
def band_power(x,sf,a,b):
    f=np.fft.rfftfreq(len(x),1/sf); p=np.abs(np.fft.rfft(x))**2; m=(f>=a)&(f<b); return float(p[m].mean()) if np.any(m) else 0.0
def load_project():
    psg,hyp=find_pair(); raw=mne.io.read_raw_edf(psg,preload=True,verbose=False); ann=mne.read_annotations(hyp); raw.set_annotations(ann,emit_warning=False)
    ch=["EEG Fpz-Cz","EEG Pz-Oz","EOG horizontal","Resp oro-nasal","EMG submental"]; names=[]
    for j,c in enumerate(ch):
        names += [f"{c}_mean",f"{c}_std",f"{c}_min",f"{c}_max",f"{c}_range",f"{c}_rms"]
        if j<2: names += [f"{c}_delta",f"{c}_theta",f"{c}_alpha",f"{c}_beta"]
    data=raw.get_data(picks=ch); sf=float(raw.info["sfreq"]); n=int(30*sf); X=[]; y=[]
    for onset,dur,label in zip(ann.onset,ann.duration,ann.description):
        if label not in ("Sleep stage 2","Sleep stage 3"): continue
        for i in range(int(dur//30)):
            s=int((onset+30*i)*sf); e=s+n
            if e>data.shape[1]: continue
            f=[]
            for j in range(5):
                v=data[j,s:e]; f += [v.mean(),v.std(),v.min(),v.max(),np.ptp(v),np.sqrt(np.mean(v*v))]
                if j<2: f += [band_power(v,sf,.5,4),band_power(v,sf,4,8),band_power(v,sf,8,13),band_power(v,sf,13,30)]
            X.append(f); y.append(0 if label=="Sleep stage 2" else 1)
    X=np.asarray(X,float); y=np.asarray(y,int); X[~np.isfinite(X)]=np.nan; med=np.nanmedian(X,axis=0); r,c=np.where(np.isnan(X)); X[r,c]=med[c]
    return X,y,names
def metrics(y,p):
    return [accuracy_score(y,p),precision_score(y,p,zero_division=0),recall_score(y,p,zero_division=0),f1_score(y,p,zero_division=0)]


In [ ]:
X,y,feature_names=load_project(); Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.3,stratify=y,random_state=42); cv=StratifiedKFold(5,shuffle=True,random_state=42)

per=Pipeline([("scale",StandardScaler()),("clf",Perceptron(random_state=42))])
per_space={"clf__penalty":[None,"l1","l2","elasticnet"],"clf__alpha":loguniform(1e-5,1e-1),"clf__eta0":loguniform(1e-3,1),"clf__max_iter":[500,1000,2000]}
per_search=RandomizedSearchCV(per,per_space,n_iter=20,scoring="f1",cv=cv,random_state=42,n_jobs=-1).fit(Xtr,ytr)

mlp=Pipeline([("scale",StandardScaler()),("clf",MLPClassifier(random_state=42,max_iter=1500))])
mlp_space={"clf__hidden_layer_sizes":[(16,),(32,),(32,16),(64,32)],"clf__activation":["relu","tanh","logistic"],"clf__alpha":loguniform(1e-5,1e-1),"clf__learning_rate_init":loguniform(1e-4,1e-1)}
mlp_search=RandomizedSearchCV(mlp,mlp_space,n_iter=20,scoring="f1",cv=cv,random_state=42,n_jobs=-1).fit(Xtr,ytr)

print(per_search.best_params_,per_search.best_score_)
print(mlp_search.best_params_,mlp_search.best_score_)


In [ ]:
models={
"Perceptron":per_search.best_estimator_,
"MLP":mlp_search.best_estimator_,
"SVM":Pipeline([("scale",StandardScaler()),("clf",SVC(probability=True,random_state=42))]),
"DecisionTree":DecisionTreeClassifier(random_state=42),
"RandomForest":RandomForestClassifier(n_estimators=300,random_state=42),
"CatBoost":CatBoostClassifier(verbose=0,random_seed=42),
"AdaBoost":AdaBoostClassifier(random_state=42),
"XGBoost":XGBClassifier(n_estimators=300,max_depth=3,learning_rate=.05,subsample=.9,colsample_bytree=.9,eval_metric="logloss",random_state=42),
"NaiveBayes":GaussianNB()
}
rows=[]
for name,m in models.items():
    t=time.perf_counter(); m.fit(Xtr,ytr); p=m.predict(Xte); dt=time.perf_counter()-t; rows.append([name,*metrics(yte,p),dt])
results=pd.DataFrame(rows,columns=["Model","Accuracy","Precision","Recall","F1","Time_s"]).sort_values("F1",ascending=False)
display(results); results.to_csv("lab09_classifier_comparison.csv",index=False)


In [ ]:
pipeline=Pipeline([("scale",StandardScaler()),("clf",SVC(probability=True,random_state=42))]); pipeline.fit(Xtr,ytr); p=pipeline.predict(Xte)
print(metrics(yte,p))


In [ ]:
explainer=LimeTabularExplainer(Xtr,feature_names=feature_names,class_names=["N2","N3"],mode="classification",discretize_continuous=True)
exp=explainer.explain_instance(Xte[0],pipeline.predict_proba,num_features=10)
lime_table=pd.DataFrame(exp.as_list(),columns=["Feature","Contribution"]); display(lime_table); lime_table.to_csv("lab09_lime_explanation.csv",index=False)
exp.as_pyplot_figure(); plt.tight_layout(); plt.show()
